# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
df = pd.read_csv('data/AviationData.csv')
df.head()
    


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_824\1901205522.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv',encoding="cp1252")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [5]:
df.isnull().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [8]:
df.shape

(88889, 31)

In [11]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [72]:
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

In [80]:
relevant_cols = [
    'Make', 'Model', 'Aircraft.damage',
    'Aircraft.Category', 'Country',
    'Total.Fatal.Injuries', 'Total.Serious.Injuries',
    'Weather.Condition', 'Broad.phase.of.flight'
]

df[relevant_cols].head()

,Make,Model,Aircraft.damage,Aircraft.Category,Country,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition,Broad.phase.of.flight
0,Stinson,108-3,Destroyed,NaN,United States,2.0,0.0,UNK,Cruise
1,Piper,PA24-180,Destroyed,NaN,United States,4.0,0.0,UNK,Unknown
2,Cessna,172M,Destroyed,NaN,United States,3.0,NaN,IMC,Cruise
3,Rockwell,112,Destroyed,NaN,United States,2.0,0.0,IMC,Cruise
4,Cessna,501,Destroyed,NaN,United States,1.0,2.0,VMC,Approach


In [81]:
df[relevant_cols].info()
df[relevant_cols].isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Make                    88826 non-null  object 
 1   Model                   88797 non-null  object 
 2   Aircraft.damage         85695 non-null  object 
 3   Aircraft.Category       32287 non-null  object 
 4   Country                 88663 non-null  object 
 5   Total.Fatal.Injuries    77488 non-null  float64
 6   Total.Serious.Injuries  76379 non-null  float64
 7   Weather.Condition       84397 non-null  object 
 8   Broad.phase.of.flight   61724 non-null  object 
dtypes: float64(2), object(7)
memory usage: 6.1+ MB


Make                         63
Model                        92
Aircraft.damage            3194
Aircraft.Category         56602
Country                     226
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Weather.Condition          4492
Broad.phase.of.flight     27165
dtype: int64

In [13]:
[col.strip() for col in df]

['Event.Id',
 'Investigation.Type',
 'Accident.Number',
 'Event.Date',
 'Location',
 'Country',
 'Latitude',
 'Longitude',
 'Airport.Code',
 'Airport.Name',
 'Injury.Severity',
 'Aircraft.damage',
 'Aircraft.Category',
 'Registration.Number',
 'Make',
 'Model',
 'Amateur.Built',
 'Number.of.Engines',
 'Engine.Type',
 'FAR.Description',
 'Schedule',
 'Purpose.of.flight',
 'Air.carrier',
 'Total.Fatal.Injuries',
 'Total.Serious.Injuries',
 'Total.Minor.Injuries',
 'Total.Uninjured',
 'Weather.Condition',
 'Broad.phase.of.flight',
 'Report.Status',
 'Publication.Date']

In [19]:
# Inspect important columns
cols = ['Aircraft.Category', 'Purpose.of.flight', 'Amateur.Built', 'Number.of.Engines']
print(df[cols].head())

# Filter to airplane accidents
df = df[df['Aircraft.Category'].fillna('').str.contains('Airplane', case=False)]

# Keep non-amateur aircraft
df = df[df['Amateur.Built'] == 'No']

# Remove rows with missing engines
df = df[df['Number.of.Engines'].notna()]

print(f"Remaining rows: {df.shape[0]}")

   Aircraft.Category Purpose.of.flight Amateur.Built  Number.of.Engines
5           Airplane               NaN            No                2.0
7           Airplane          Personal            No                1.0
8           Airplane          Business            No                2.0
12          Airplane          Personal            No                1.0
13          Airplane          Personal            No                1.0
Remaining rows: 21866


In [ ]:
df_fill= [col for col in df if col is NaN fillna('unkwnown')]

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [20]:
# Fill missing injury counts
injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Estimate total passengers
df['Estimated.Passengers'] = (
    df['Total.Fatal.Injuries'] +
    df['Total.Serious.Injuries'] +
    df['Total.Minor.Injuries'] +
    df['Total.Uninjured']
)

# Remove divide-by-zero rows
df = df[df['Estimated.Passengers'] > 0]

# Create injury metrics
df['Fatal.Serious.Count'] = (
    df['Total.Fatal.Injuries'] +
    df['Total.Serious.Injuries']
)

df['Fatal.Serious.Fraction'] = (
    df['Fatal.Serious.Count'] /
    df['Estimated.Passengers']
)

df[['Estimated.Passengers', 'Fatal.Serious.Fraction']].head()

,Estimated.Passengers,Fatal.Serious.Fraction
5,45.0,0.0
7,2.0,0.0
8,2.0,0.0
12,1.0,0.0
13,1.0,1.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [22]:
# Inspect aircraft damage
print(df['Aircraft.damage'].value_counts(dropna=False))

# Standardize labels
df['Aircraft.damage'] = (
    df['Aircraft.damage']
    .fillna('Unknown')
    .str.title()
)
# Create binary destruction variable
df['Destroyed'] = np.where(
    df['Aircraft.damage'] == 'Destroyed',
    1,
    0
)

df[['Aircraft.damage', 'Destroyed']].head()

Aircraft.damage
Substantial    17892
Destroyed       2526
Unknown          605
Minor            534
Name: count, dtype: int64


,Aircraft.damage,Destroyed
5,Substantial,0
7,Substantial,0
8,Substantial,0
12,Destroyed,1
13,Destroyed,1


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [23]:
print(df['Make'].value_counts().head(20))

# Standardize make labels
df['Make'] = (
    df['Make']
    .astype(str)
    .str.upper()
    .str.strip()
)

# Keep makes with at least 50 records
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index

df = df[df['Make'].isin(valid_makes)]

print(df['Make'].nunique())

Make
CESSNA                4408
Cessna                3439
PIPER                 2598
Piper                 1826
BEECH                  903
Beech                  609
BOEING                 397
MOONEY                 220
CIRRUS DESIGN CORP     216
AIR TRACTOR INC        203
Mooney                 174
Grumman                167
Boeing                 152
BELLANCA               150
AERONCA                146
MAULE                  137
Air Tractor            127
Bellanca               122
LUSCOMBE                92
STINSON                 91
Name: count, dtype: int64
36


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [24]:
df['Model'] = df['Model'].fillna('UNKNOWN')

# Standardize model names
df['Model'] = (
    df['Model']
    .astype(str)
    .str.upper()
    .str.strip()
)

# Create unique plane type identifier
df['Plane.Type'] = df['Make'] + ' ' + df['Model']

print(df[['Make', 'Model', 'Plane.Type']].head())

                 Make   Model             Plane.Type
5   MCDONNELL DOUGLAS     DC9  MCDONNELL DOUGLAS DC9
7              CESSNA     140             CESSNA 140
8              CESSNA    401B            CESSNA 401B
12           BELLANCA  17-30A        BELLANCA 17-30A
13             CESSNA   R172K           CESSNA R172K


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [25]:
cat_cols = [
    'Engine.Type',
    'Weather.Condition',
    'Purpose.of.flight',
    'Broad.phase.of.flight'
]

for col in cat_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head())

# Standardize categories
for col in cat_cols:
    df[col] = (
        df[col]
        .fillna('Unknown')
        .astype(str)
        .str.title()
        .str.strip()
    )

# Numeric cleanup
df['Number.of.Engines'] = pd.to_numeric(
    df['Number.of.Engines'],
    errors='coerce'
)

df['Number.of.Engines'] = (
    df['Number.of.Engines']
    .fillna(df['Number.of.Engines'].median())
)

df.head()


Engine.Type
Engine.Type
Reciprocating    15273
NaN               1440
Turbo Prop         930
Turbo Fan          560
Turbo Jet           72
Name: count, dtype: int64

Weather.Condition
Weather.Condition
VMC    16235
IMC     1096
NaN      790
Unk      124
UNK       56
Name: count, dtype: int64

Purpose.of.flight
Purpose.of.flight
Personal              11062
Instructional          2709
NaN                    1363
Aerial Application      787
Business                640
Name: count, dtype: int64

Broad.phase.of.flight
Broad.phase.of.flight
NaN         13114
Landing      1879
Takeoff       983
Cruise        698
Approach      502
Name: count, dtype: int64


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Estimated.Passengers,Fatal.Serious.Count,Fatal.Serious.Fraction,Destroyed,Plane.Type
5,20170710X52551,Accident,NYC79AA106,1979-09-17,"BOSTON, MA",United States,42.445277,-70.758333,NaN,NaN,...,44.0,Vmc,Climb,Probable Cause,19-09-2017,45.0,0.0,0.0,0,MCDONNELL DOUGLAS DC9
7,20020909X01562,Accident,SEA82DA022,1982-01-01,"PULLMAN, WA",United States,NaN,NaN,NaN,BLACKBURN AG STRIP,...,2.0,Vmc,Takeoff,Probable Cause,01-01-1982,2.0,0.0,0.0,0,CESSNA 140
8,20020909X01561,Accident,NYC82DA015,1982-01-01,"EAST HANOVER, NJ",United States,NaN,NaN,N58,HANOVER,...,2.0,Imc,Landing,Probable Cause,01-01-1982,2.0,0.0,0.0,0,CESSNA 401B
12,20020917X02148,Accident,FTW82FRJ07,1982-01-02,"HOMER, LA",United States,NaN,NaN,NaN,NaN,...,0.0,Imc,Cruise,Probable Cause,02-01-1983,1.0,0.0,0.0,1,BELLANCA 17-30A
13,20020917X02134,Accident,FTW82FRA14,1982-01-02,"HEARNE, TX",United States,NaN,NaN,T72,HEARNE MUNICIPAL,...,0.0,Imc,Takeoff,Probable Cause,02-01-1983,1.0,1.0,1.0,1,CESSNA R172K


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [26]:
# Check missing value percentages
missing_pct = df.isna().mean().sort_values(ascending=False)

# Drop columns with more than 60% missing values
drop_cols = missing_pct[missing_pct > 0.60].index

print("Dropped columns:")
print(drop_cols)

df = df.drop(columns=drop_cols)

print(df.shape)

Dropped columns:
Index(['Schedule'], dtype='object')
(18301, 35)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [27]:
df.to_csv('cleaned_aviation_accidents.csv', index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
